# Unit 3 exercises: bag-of-words, TF-IDF cosine, and BM25

Run the recsys data generators before you begin. These exercises rebuild the lexical path from the lesson -- a book as a bag of keyword tokens, TF-IDF with cosine similarity, then Okapi BM25 -- reveal the `bookrec` implementation piece by piece, and score it honestly on the Unit 1 scoreboard. A reader's query is always the keyword tokens of the books in their positive `train` history (the `seen` set); `val` is the sealed holdout we grade against. Each answer is left for you to complete; no solutions live here.

In [ ]:
import json
from collections import Counter

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])


def tokenize(text):
    return text.split()


# each reader's positive train items == the scoreboard's `seen` set
train_positive = interactions[(interactions["split"] == "train") & (interactions["label"] == 1)]
seen_by = train_positive.groupby("reader_id")["item_id"].apply(set).to_dict()


def profile_of(reader_id):
    # the reader's query: keyword tokens of every book they have read, repeats kept
    return [token for item_id in sorted(seen_by[reader_id]) for token in tokenize(keywords[item_id])]

## Exercise 1 - Build the bag-of-words view: vocabulary and document frequency

A bag-of-words model keeps *which* tokens a document contains and *how many times*, and nothing else. Tokenize every book's keyword document with `tokenize` (whitespace split, repeats kept) into `tokens_of`, a `dict[int, list[str]]`. Collect the **vocabulary** (the sorted set of distinct tokens across the catalog) and compute the **document frequency** $\mathrm{df}_t$ of every token -- the number of *documents* containing it at least once -- by adding each document's *set* of tokens to a `Counter`. Record the vocabulary size, the single most widespread token and its `df`, how many tokens appear in exactly one book, and the **smallest** `df` in the corpus (on this corpus the single-book count may well be 0 -- the keyword generator keeps every token in dozens of books -- so the smallest `df` is what shows a 0 is a fact about the data, not a bug). Then verify a bookkeeping identity: the sum of all `df` values must equal the sum over documents of the number of *distinct* tokens in that document (every (document, token) pair with tf > 0 is counted exactly once on each side).

In [ ]:
tokens_of = None            # TODO: {item_id: tokenize(keywords[item_id]) for every catalog id}
vocab = None                # TODO: sorted list of distinct tokens across all documents
vocab_size = None           # TODO: len(vocab)

df_counter = Counter()      # TODO: update it with set(tokens) for each document
most_widespread_token = None  # TODO: the token with the largest df
most_widespread_df = None     # TODO: its df
singleton_tokens = None       # TODO: how many tokens have df == 1 (may be 0 on this corpus)
smallest_df = None            # TODO: min of the df values, so a 0 above is not mistaken for a bug

df_total = None             # TODO: sum of df over the vocabulary
distinct_total = None       # TODO: sum over documents of len(set(tokens))
identity_holds = None       # TODO: bool - df_total == distinct_total

## Exercise 2 - TF-IDF weights and cosine similarity for a reader's profile, then reveal the library

Build the dense document-term matrix `tf` of shape `(n_docs, n_terms)` in sorted `catalog_ids` row order and alphabetical `vocab` column order, and check that its row sums are the document lengths and its column-wise non-zero counts equal your Exercise 1 `df`. Compute the smoothed IDF the lesson used,

$$\mathrm{idf}_t = \log\!\left(1 + \frac{N - \mathrm{df}_t + 0.5}{\mathrm{df}_t + 0.5}\right),$$

weight `tf * idf`, and L2-normalize every row so each document vector has unit length. Now take **reader 7**: their query is `profile_of(7)`. Build the query vector the same way (token counts times `idf`), compute the cosine of the query against every document (with unit-length rows this is one matrix-vector product divided by the query's norm), and use `bookrec.rank(..., n=5, exclude=seen)` on `bookrec.Candidate` objects to get the top-5 unseen books.

Then reveal the package: `bookrec.BM25Index(keywords)` stores `vocab`, `tf`, `df`, `idf`, `doc_len`, `avgdl` with rows in sorted `item_id` order, `bookrec.tfidf_matrix(index)` returns the L2-normalized rows, and `bookrec.cosine_similarity(query_vector, matrix)` the cosine against each row. The library numbers its vocabulary in first-seen order, so build the column `permutation` that maps your alphabetical columns onto the library's, and confirm that the library's TF-IDF matrix and cosine scores match yours to floating-point precision.

In [ ]:
n_docs, n_terms = len(catalog_ids), len(vocab)
col_of = {token: col for col, token in enumerate(vocab)}

tf = np.zeros((n_docs, n_terms))   # TODO: fill tf[row, col_of[token]] with real counts
doc_lengths = None       # TODO: np.array of each document's token count, in catalog_ids order
df = None                # TODO: column-wise count of non-zero entries in tf
tf_checks_pass = None    # TODO: bool - row sums == doc_lengths AND df matches Exercise 1's df_counter

idf = None               # TODO: the smoothed idf formula above, as an array over vocab
tfidf_by_hand = None     # TODO: (tf * idf) with every row L2-normalized to unit length

reader_id = 7
seen = seen_by[reader_id]
profile_tokens = profile_of(reader_id)
query = None             # TODO: token counts of profile_tokens in vocab order, times idf
cosine_by_hand = None    # TODO: cosine of query against every (unit-length) document row
top5_tfidf = None        # TODO: bookrec.rank([Candidate(item_id, score, "tfidf-by-hand") ...], n=5, exclude=seen)
top5_tfidf_ids = []      # TODO: the item ids of top5_tfidf

# reveal the library
index = None             # TODO: bookrec.BM25Index(keywords)
permutation = None       # TODO: np.array([index.vocab[token] for token in vocab])
tfidf_lib = None         # TODO: bookrec.tfidf_matrix(index)
query_lib = None         # TODO: the same query vector in the library's column order (times index.idf)
cosine_lib = None        # TODO: bookrec.cosine_similarity(query_lib, tfidf_lib)
library_matches = None   # TODO: bool - tfidf_lib[:, permutation] ~ tfidf_by_hand AND cosine_lib ~ cosine_by_hand

## Exercise 3 - Fit and register LexicalRetrievalPath, then read the scoreboard vs the floor

The package ships BM25 retrieval as `bookrec.LexicalRetrievalPath(keywords)`: it takes the keyword corpus at construction (like `RandomRetrievalPath(item_ids)`), so `fit(rows, catalog=catalog_ids)` keeps the exact `RetrievalPath` protocol. Fit it, fit a `bookrec.PopularityRetrievalPath()` the same way, build a `bookrec.RandomRetrievalPath(catalog_ids, seed=0)` floor, and register all three in a fresh `bookrec.PathRegistry`. Score each on the `val` scoreboard with `bookrec.run_validation_scoreboard(path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)`. Record each `hit_rate_at_k`, the number of readers scored, the ratio lexical/random, and the ratio lexical/popularity.

Then write two or three honest sentences. "Beats the floor" here means hit@10 more than 10x the seeded random floor; say what that does and does **not** establish. The lesson was careful: lexical retrieval is a strong **content candidate source**, in the same family as simple genre matching (the keyword text correlates with genres), and it should be expected to sit *below* the collaborative paths of later units. Do not describe this number as BM25 beating TF-IDF cosine -- the lesson measured that question separately and found the two to be peers on this corpus.

In [ ]:
lexical = None           # TODO: bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
popularity = None        # TODO: bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
random_floor = None      # TODO: bookrec.RandomRetrievalPath(catalog_ids, seed=0)
registry = None          # TODO: bookrec.PathRegistry(); register all three paths


def score(path):
    return None          # TODO: bookrec.run_validation_scoreboard(path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)


lexical_board = None     # TODO: score(lexical)
pop_board = None         # TODO: score(popularity)
random_board = None      # TODO: score(random_floor)
readers_scored = None    # TODO: lexical_board.readers
lexical_hit = None       # TODO: lexical_board.hit_rate_at_k
vs_floor = None          # TODO: lexical_hit / random_board.hit_rate_at_k  (expect > 10)
vs_popularity = None     # TODO: lexical_hit / pop_board.hit_rate_at_k
beats_floor_explanation = "TODO"  # two or three honest sentences

## Exercise 4 - The b length-normalization effect on a long and a short document

BM25's $b$ controls how much document length enters the denominator: at $b = 0$ length is ignored and a long bag gets full credit for every matched term; at $b = 1$ a document twice the average length needs its terms about twice as often to score the same. Find the **longest** and the **shortest** keyword document in the catalog (break ties by smaller `item_id`) and record their lengths and the corpus average `avgdl`. Build two indexes, `bookrec.BM25Index(keywords, b=0.0)` and `bookrec.BM25Index(keywords, b=1.0)`, and score reader 7's `profile_tokens` with each. For both documents report the BM25 score at each $b$ and the document's **rank** among all catalog books (1 = best, use a stable descending sort of the score array). Then measure the bias over the whole top-10 the way the lesson did for one reader: the mean document length of reader 7's top-10 unseen books at $b = 0$ versus $b = 1$ (use `bookrec.rank(..., n=10, exclude=seen)`).

In one sentence, say which direction the long document's rank moves as $b$ goes from 0 to 1 and why that is exactly the effect $b$ exists to produce.

In [ ]:
lengths_by_id = None     # TODO: {item_id: len(tokenize(keywords[item_id]))}
long_id = None           # TODO: item_id of the longest document (smallest id on ties)
short_id = None          # TODO: item_id of the shortest document (smallest id on ties)
long_len, short_len = None, None  # TODO: their lengths
avgdl = None             # TODO: corpus mean document length (compare with index.avgdl)

index_b0 = None          # TODO: bookrec.BM25Index(keywords, b=0.0)
index_b1 = None          # TODO: bookrec.BM25Index(keywords, b=1.0)
scores_b0 = None         # TODO: index_b0.score(profile_tokens)
scores_b1 = None         # TODO: index_b1.score(profile_tokens)


def rank_of(item_id, scores):
    return None          # TODO: 1-based position of item_id in a stable descending sort of scores


long_score_b0, long_score_b1 = None, None    # TODO
short_score_b0, short_score_b1 = None, None  # TODO
long_rank_b0, long_rank_b1 = None, None      # TODO: rank_of(long_id, ...)
short_rank_b0, short_rank_b1 = None, None    # TODO: rank_of(short_id, ...)

mean_top10_len_b0 = None  # TODO: mean length of reader 7's top-10 unseen books at b = 0
mean_top10_len_b1 = None  # TODO: the same at b = 1
b_effect_explanation = "TODO"  # one sentence

## Exercise 5 - Compute one BM25 score by hand and match BM25Index.score

For a query $q$ (a multiset of tokens) and a document $d$,

$$\mathrm{BM25}(q, d) = \sum_{t \in q} \mathrm{idf}_t \cdot \frac{\mathrm{tf}_{d,t}\,(k_1 + 1)}{\mathrm{tf}_{d,t} + k_1\left(1 - b + b\,\dfrac{|d|}{\mathrm{avgdl}}\right)},$$

where a query token repeated $n$ times adds its contribution $n$ times. Take the default `index = bookrec.BM25Index(keywords)` ($k_1 = 1.2$, $b = 0.75$) and the single document `long_id` from Exercise 4. Using *only* the index's stored arrays -- `index.tf`, `index.idf`, `index.vocab`, `index.doc_len`, `index.avgdl`, `index.k1`, `index.b` -- and a `Counter` of reader 7's `profile_tokens`, loop over the distinct query tokens and accumulate the formula term by term for that one document (skip query tokens that are not in the vocabulary, exactly as the library does). Record the hand score, the library's `index.score(profile_tokens)[row]` for the same document, and whether they agree to floating-point tolerance. Also record how many distinct query tokens actually contributed (tf > 0 in that document) and, of those, how many had tf = 1. Then widen the tally: rank every book by `index.score(profile_tokens)` with `bookrec.rank(..., n=10, exclude=seen)`, and over those top-10 recommendations compute what fraction of the matched (query token, book) pairs -- pairs with tf > 0 -- have tf = 1. That fraction is the reason the lesson called $k_1$ saturation a small effect on this corpus: when almost every matched term appears once, the saturation curve has nothing to shape.

In [ ]:
index = bookrec.BM25Index(keywords)
row = None               # TODO: the row of long_id in index.item_ids
query_tf = None          # TODO: Counter of profile_tokens restricted to tokens in index.vocab

length_term = None       # TODO: k1 * (1 - b + b * index.doc_len[row] / index.avgdl) for this document
bm25_by_hand = 0.0       # TODO: accumulate q_count * idf_t * tf * (k1 + 1) / (tf + length_term) over query tokens
bm25_lib = None          # TODO: index.score(profile_tokens)[row]
hand_matches_library = None  # TODO: bool - agree to tolerance

contributing_terms = None    # TODO: distinct query tokens with tf > 0 in this document
contributing_tf1 = None      # TODO: how many of those have tf == 1

top10_default = None         # TODO: bookrec.rank over index.score(profile_tokens) as Candidates, n=10, exclude=seen
matched_pairs = None         # TODO: list of the tf values > 0 over (query token, top-10 book) pairs
share_tf1_top10 = None       # TODO: fraction of matched_pairs equal to 1

## Exercise 6 - Reader-dependent retrieval: shipped path vs by-hand ranking, exclusions, and no history

`LexicalRetrievalPath.retrieve(query, context, k)` identifies the reader through `context["seen"]`, not the `query` id: it concatenates the seen books' keyword bags into the query, scores every document with BM25, drops the seen books, and returns the top `k`. Confirm this three ways with the `lexical` path fitted in Exercise 3. First, retrieve reader 7's top-5 and check it equals the top-5 you get by wrapping `index.score(profile_tokens)` in `bookrec.Candidate` objects and calling `bookrec.rank(..., n=5, exclude=seen)` (same item ids, same order). Second, check that none of reader 7's seen books appear in the result and that reader 7 and reader 11 receive *different* top-5 lists -- this is the first path in the registry whose output depends on the reader. Third, retrieve for a reader with an empty `seen` set and confirm the answer is `[]`; in one sentence, say why that is the honest answer and which Unit 2 path steps in for such a reader.

In [ ]:
shipped_top5 = None      # TODO: lexical.retrieve(query=reader_id, context={"seen": seen}, k=5)
shipped_ids = []         # TODO: item ids of shipped_top5
by_hand_top5 = None      # TODO: bookrec.rank([Candidate(i, score, "bm25") for i, score in zip(index.item_ids, index.score(profile_tokens))], n=5, exclude=seen)
by_hand_ids = []         # TODO: item ids of by_hand_top5
same_ranking = None      # TODO: bool - shipped_ids == by_hand_ids

no_seen_leaked = None    # TODO: bool - no id in shipped_ids is in seen
other_reader = 11
other_ids = []           # TODO: item ids of lexical.retrieve(other_reader, {"seen": seen_by[other_reader]}, 5)
reader_dependent = None  # TODO: bool - other_ids != shipped_ids

no_history = None        # TODO: lexical.retrieve(query=999_999, context={"seen": set()}, k=5)
no_history_is_empty = None  # TODO: bool - no_history == []
cold_start_explanation = "TODO"  # one sentence

## Exercise 7 - Challenge: the two k1 limits -- linear tf and binary presence

The saturating term $\frac{\mathrm{tf}\,(k_1 + 1)}{\mathrm{tf} + k_1 L}$ with $L = 1 - b + b\,|d|/\mathrm{avgdl}$ has two instructive limits. Show algebraically that as $k_1 \to \infty$ it tends to $\mathrm{tf} / L$ (so at $b = 0$ BM25 becomes the *linear* sum $\sum_t q_t \cdot \mathrm{idf}_t \cdot \mathrm{tf}_{d,t}$ -- raw TF-IDF without the cosine), and that as $k_1 \to 0$ it tends to $1$ whenever $\mathrm{tf} > 0$ (so BM25 becomes $\sum_t q_t \cdot \mathrm{idf}_t \cdot [\mathrm{tf}_{d,t} > 0]$: **binary term presence**, length ignored). Verify both numerically on reader 7's profile: build `bookrec.BM25Index(keywords, k1=1e6, b=0.0)` and compare its scores against the linear sum computed from `index.tf` and `index.idf` (use a relative tolerance of about $10^{-3}$); build `bookrec.BM25Index(keywords, k1=0.0)` and compare against the presence sum exactly.

Then put the binary-presence limit on the scoreboard: score `bookrec.LexicalRetrievalPath(keywords, k1=0.0, name="bm25-k1-0")` on `val` exactly as in Exercise 3 and record its hit@10 next to the default. The lesson said every BM25 setting lands in one narrow band here; you should find the $k_1 = 0$ limit sits at the top of, or level with, that band. In two sentences explain *why* binary presence is about as good as it gets on 26-token keyword bags where nearly every matched term has tf = 1 -- and what kind of corpus would make the count, and hence $k_1$, matter.

In [ ]:
linear_limit_derivation = "TODO"     # one or two sentences of algebra for k1 -> infinity
presence_limit_derivation = "TODO"   # one or two sentences of algebra for k1 -> 0

index_big_k1 = None      # TODO: bookrec.BM25Index(keywords, k1=1e6, b=0.0)
linear_scores = None     # TODO: sum over query tokens of q_count * idf_t * tf[:, col] from `index`
big_k1_is_linear = None  # TODO: bool - np.allclose(index_big_k1.score(profile_tokens), linear_scores, rtol=1e-3)

index_k1_zero = None     # TODO: bookrec.BM25Index(keywords, k1=0.0)
presence_scores = None   # TODO: sum over query tokens of q_count * idf_t * (tf[:, col] > 0)
k1_zero_is_presence = None  # TODO: bool - np.allclose(index_k1_zero.score(profile_tokens), presence_scores)

k1_zero_board = None     # TODO: score(bookrec.LexicalRetrievalPath(keywords, k1=0.0, name="bm25-k1-0").fit(rows, catalog=catalog_ids))
k1_zero_hit = None       # TODO: k1_zero_board.hit_rate_at_k
default_hit = None       # TODO: lexical_board.hit_rate_at_k (Exercise 3)
why_presence_suffices = "TODO"  # two sentences

## Exercise 8 - Challenge: why a rare keyword dominates a match

IDF is the term that *can* let one word outvote twenty. First explain from the formula why, in general, the ratio of the rarest token's idf to the commonest token's idf can be enormous: with $N$ documents, a token in a single book ($\mathrm{df} = 1$) earns about $\log N$, while a token in nearly every book ($\mathrm{df} \approx N$) earns $\log(1 + \text{small}) \approx 0$, so the ratio is unbounded. Then measure it here: among the distinct tokens of reader 7's profile, find the token with the **highest** `idf` (the rarest) and the one with the **lowest** (the most widespread), and record their `df` and `idf` values together with the ratio of the two idf values. Expect a modest number on this corpus -- the keyword generator keeps every token in dozens of books, so `df` spans a narrow range and no token is anywhere near the formula's extremes -- and say in your explanation why the measured ratio is far from the general bound.

Now show the consequence on one match. Take the default `index` and the top-1 unseen book from Exercise 6 (`shipped_ids[0]`). Decompose its BM25 score into per-token contributions -- for each distinct query token, `q_count * idf_t * saturated_tf` for that document, using the stored arrays as in Exercise 5 -- and confirm the contributions sum to `index.score(profile_tokens)` for that book. Record the token with the largest contribution, its `df`, its share of the total score, and the share of the total score carried by the top-3 contributing tokens together. Finally, build a two-token query `[rare_token, common_token]` and score it: confirm that the top-ranked document contains the rare token, and say in one sentence what would go wrong in a recommender that scored matches by raw `tf` alone.

In [ ]:
profile_distinct = None  # TODO: sorted set of profile_tokens
rare_token = None        # TODO: the profile token with the largest index.idf
common_token = None      # TODO: the profile token with the smallest index.idf
rare_df, rare_idf = None, None      # TODO: index.df / index.idf at index.vocab[rare_token]
common_df, common_idf = None, None  # TODO: the same for common_token
idf_ratio = None         # TODO: rare_idf / common_idf
idf_ratio_explanation = "TODO"  # one or two sentences: the formula's general bound, then why this corpus is far from it

top1_id = None           # TODO: shipped_ids[0] from Exercise 6
top1_row = None          # TODO: its row in index.item_ids
contributions = {}       # TODO: {token: q_count * idf_t * saturated_tf} for every contributing query token
contributions_sum_matches = None  # TODO: bool - sum(contributions.values()) ~ index.score(profile_tokens)[top1_row]
dominant_token = None    # TODO: token with the largest contribution
dominant_df = None       # TODO: its df
dominant_share = None    # TODO: its contribution / the total score
top3_share = None        # TODO: sum of the three largest contributions / the total score

two_token_scores = None  # TODO: index.score([rare_token, common_token])
two_token_top_id = None  # TODO: item id with the highest two_token_scores (stable descending sort)
top_contains_rare = None  # TODO: bool - rare_token in tokenize(keywords[two_token_top_id])
raw_tf_failure = "TODO"  # one sentence